# Safe Driving — YOLOv8 Demo

This notebook presents the final workflow of the Safe Driving computer vision project.

The system uses YOLOv8 to detect:
- Mobile phone
- Seatbelt
- No seatbelt
- Windshield

The project also includes video inference and temporal smoothing to reduce unstable frame-by-frame predictions.

## 1. Imports

In [ ]:
from pathlib import Path
from collections import deque

import cv2
from ultralytics import YOLO

## 2. Configuration

The trained YOLOv8 model is loaded from the project directory.

The model weights (`.pt`) and test videos are excluded from GitHub using `.gitignore`.

In [ ]:
PROJECT_DIR = Path(r"C:\Users\user\Desktop\Safe-driving")

MODEL_PATH = (
    PROJECT_DIR
    / "runs"
    / "safe_driving_v3"
    / "weights"
    / "best.pt"
)

VIDEO_PATH = PROJECT_DIR / "video_test.mp4"

model = YOLO(str(MODEL_PATH))

print("Model loaded:", MODEL_PATH)
print("Classes:", model.names)

## 3. Image Detection

YOLOv8 is applied to a test image to detect the objects learned during training.

In [ ]:
IMAGE_PATH = PROJECT_DIR / "test_frame_reelle.jpg"

results = model.predict(
    source=str(IMAGE_PATH),
    conf=0.25,
    verbose=False
)

annotated = results[0].plot()

output_path = PROJECT_DIR / "results" / "demo_detection.jpg"
cv2.imwrite(str(output_path), annotated)

print("Detection completed.")
print("Result saved to:", output_path)

## 4. Temporal Smoothing

Frame-by-frame predictions can fluctuate.

A sliding window of recent detections is therefore used to confirm the driver's status and reduce unstable predictions.

In [ ]:
WINDOW_SIZE = 15
MIN_POSITIVE_FRAMES = 9

seatbelt_history = deque(maxlen=WINDOW_SIZE)
no_seatbelt_history = deque(maxlen=WINDOW_SIZE)


def update_status(
    seatbelt_detected,
    no_seatbelt_detected,
    mobile_detected
):
    seatbelt_history.append(seatbelt_detected)
    no_seatbelt_history.append(no_seatbelt_detected)

    seatbelt_confirmed = (
        sum(seatbelt_history) >= MIN_POSITIVE_FRAMES
    )

    no_seatbelt_confirmed = (
        sum(no_seatbelt_history) >= MIN_POSITIVE_FRAMES
    )

    if mobile_detected:
        return "WARNING - MOBILE DETECTED"

    elif no_seatbelt_confirmed and not seatbelt_confirmed:
        return "WARNING - NO SEATBELT"

    elif seatbelt_confirmed:
        return "SAFE - SEATBELT CONFIRMED"

    return "CHECKING SEATBELT"

## 5. Video Inference

The trained YOLOv8 model processes the video frame by frame.

Temporal smoothing is applied to obtain a more stable driver status.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError(
        f"Unable to open video: {VIDEO_PATH}"
    )

frame_count = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    result = model.predict(
        source=frame,
        conf=0.20,
        verbose=False
    )[0]

    seatbelt = False
    no_seatbelt = False
    mobile = False

    for cls, conf in zip(
        result.boxes.cls,
        result.boxes.conf
    ):

        class_name = model.names[int(cls)]
        confidence = float(conf)

        if class_name == "seatbelt" and confidence >= 0.25:
            seatbelt = True

        elif class_name == "no_seatbelt" and confidence >= 0.30:
            no_seatbelt = True

        elif class_name == "mobile" and confidence >= 0.30:
            mobile = True

    status = update_status(
        seatbelt,
        no_seatbelt,
        mobile
    )

    frame_count += 1

cap.release()

print(
    f"Video inference completed: "
    f"{frame_count} frames processed."
)

## 6. Final Model Results

The final V3 model was trained using YOLOv8n with an input size of 640×640.

| Class | Precision | Recall | mAP50 |
|---|---:|---:|---:|
| Mobile | 75.6% | 81.6% | 79.4% |
| No Seatbelt | 46.7% | 67.7% | 48.2% |
| Seatbelt | 93.1% | 95.1% | 96.8% |
| Windshield | 99.5% | 100.0% | 99.5% |
| **Overall** | **78.7%** | **86.1%** | **81.0%** |

### Training configuration

- Model: YOLOv8n
- Image size: 640 × 640
- Epochs: up to 50
- Batch size: 16
- Framework: Ultralytics
- Dataset: Roboflow

## 7. Conclusion

This project demonstrates a complete driver-monitoring pipeline using:

- YOLOv8 for object detection
- OpenCV for video processing
- Temporal smoothing for more stable decisions

The main objective is to detect driver safety-related elements such as mobile phone usage and seatbelt status.

Future improvements can focus on:
- improving `no_seatbelt` detection
- reducing false detections
- improving robustness in different lighting conditions
- optimizing real-time performance